# DSA 05: Graphs
## Networks, Relationships, and Traversals

**What you'll learn:**
- Graph representations (adjacency list, adjacency matrix)
- BFS (Breadth-First Search) -- shortest path, level by level
- DFS (Depth-First Search) -- explore all paths, connected components
- Cycle detection (directed and undirected)
- Topological sort -- dependency ordering
- Union-Find (Disjoint Set Union) -- connected components
- Dijkstra's algorithm -- shortest path with weights
- 15+ practice problems with solutions

**Prerequisites:** DSA_04 (Trees & Heaps)
**Estimated Time:** 6-7 hours

---

> Graphs are everywhere: social networks, road maps, pipeline dependencies,
> Spark DAGs, Kafka topic topologies. Master BFS and DFS, and you can
> solve most graph problems.

---

---
# Section 1: Graph Basics

## What is a Graph?

A graph is a set of **nodes (vertices)** connected by **edges**.

```
Undirected:              Directed (digraph):
  A --- B --- C            A -> B -> C
  |     |                  |         |
  D --- E                  v         v
                           D         E
```

## Graph Types

| Type | Description | Example |
|------|-------------|---------|
| **Undirected** | Edges go both ways | Social network (friendships) |
| **Directed** | Edges have direction | Twitter (follows), pipeline DAGs |
| **Weighted** | Edges have costs/distances | Road map, network latency |
| **Cyclic** | Contains cycles | Most real-world graphs |
| **DAG** | Directed Acyclic Graph | Pipeline dependencies, Spark DAG |
| **Connected** | Path exists between all pairs | Single component |

## DE Connection -- Graphs Are Everywhere

| DE Concept | Graph Underneath |
|-----------|-----------------|
| Airflow/Lakeflow DAG | Directed acyclic graph of tasks |
| Spark execution plan | DAG of transformations |
| Data lineage | Directed graph (table -> table) |
| Kafka topic topology | Directed graph (producers -> topics -> consumers) |
| Foreign key relationships | Undirected graph of tables |
| Network routing | Weighted directed graph |

## Two Representations

### 1. Adjacency List (most common -- use for sparse graphs)
```python
graph = {
    "A": ["B", "D"],
    "B": ["A", "C", "E"],
    "C": ["B"],
    "D": ["A", "E"],
    "E": ["B", "D"]
}
```

### 2. Adjacency Matrix (use for dense graphs)
```python
#     A  B  C  D  E
# A [[0, 1, 0, 1, 0],
# B  [1, 0, 1, 0, 1],
# C  [0, 1, 0, 0, 0],
# D  [1, 0, 0, 0, 1],
# E  [0, 1, 0, 1, 0]]
```

In [ ]:
# Graph representations
from collections import defaultdict, deque

print("GRAPH REPRESENTATIONS")
print("=" * 60)

# Adjacency List (most common)
print("\n1. Adjacency List (dict of lists):")
graph = {
    "A": ["B", "D"],
    "B": ["A", "C", "E"],
    "C": ["B"],
    "D": ["A", "E"],
    "E": ["B", "D"],
}
for node, neighbors in graph.items():
    print(f"   {node} -> {neighbors}")

# Building from edge list
print("\n2. Building from edge list:")
edges = [("A", "B"), ("A", "D"), ("B", "C"), ("B", "E"), ("D", "E")]
graph2 = defaultdict(list)
for u, v in edges:
    graph2[u].append(v)
    graph2[v].append(u)  # Undirected: add both directions
print(f"   Edges: {edges}")
for node in sorted(graph2):
    print(f"   {node} -> {graph2[node]}")

# Directed graph
print("\n3. Directed graph (edges only go one way):")
directed = {
    "A": ["B", "C"],
    "B": ["D"],
    "C": ["D"],
    "D": [],
}
for node, neighbors in directed.items():
    print(f"   {node} -> {neighbors}")
print("   (This is a DAG -- like a pipeline dependency graph!)")

---
# Section 2: BFS -- Breadth-First Search

## What is BFS?

Explore the graph **level by level** -- visit all nodes at distance 1,
then all at distance 2, then distance 3, etc.

Uses a **queue** (FIFO).

```
Starting from A:
Level 0: A
Level 1: B, D (neighbors of A)
Level 2: C, E (neighbors of B and D, not yet visited)
```

## BFS Template

```python
def bfs(graph, start):
    visited = set([start])
    queue = deque([start])

    while queue:
        node = queue.popleft()
        process(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
```

## When to Use BFS

- **Shortest path** in unweighted graph
- **Level-order traversal**
- **Connected components**
- **Minimum steps/moves** problems
- Anything asking for "minimum" or "nearest" in an unweighted graph

In [ ]:
# BFS implementation
print("BFS -- Breadth-First Search")
print("=" * 60)

def bfs(graph, start):
    """
    BFS traversal. Returns order of visited nodes.
    Time: O(V + E), Space: O(V)
    """
    visited = set([start])
    queue = deque([start])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order

graph = {
    "A": ["B", "D"],
    "B": ["A", "C", "E"],
    "C": ["B"],
    "D": ["A", "E"],
    "E": ["B", "D"],
}

print(f"  Graph: {dict(graph)}")
print(f"  BFS from A: {bfs(graph, 'A')}")
print(f"  BFS from C: {bfs(graph, 'C')}")
print()
print("  BFS explores level by level:")
print("  From A: A(level 0) -> B,D(level 1) -> C,E(level 2)")

In [ ]:
# BFS Application: Shortest Path in Unweighted Graph
print("\nBFS: SHORTEST PATH (unweighted)")
print("=" * 60)

def shortest_path(graph, start, end):
    """
    Find shortest path between start and end using BFS.
    Time: O(V + E), Space: O(V)
    """
    if start == end:
        return [start]

    visited = set([start])
    queue = deque([(start, [start])])  # (node, path_so_far)

    while queue:
        node, path = queue.popleft()

        for neighbor in graph[node]:
            if neighbor not in visited:
                new_path = path + [neighbor]
                if neighbor == end:
                    return new_path
                visited.add(neighbor)
                queue.append((neighbor, new_path))

    return []  # No path exists

graph = {
    "A": ["B", "D"],
    "B": ["A", "C", "E"],
    "C": ["B", "F"],
    "D": ["A", "E"],
    "E": ["B", "D", "F"],
    "F": ["C", "E"],
}

path = shortest_path(graph, "A", "F")
print(f"  Shortest path A -> F: {path} (length {len(path) - 1})")
path = shortest_path(graph, "D", "C")
print(f"  Shortest path D -> C: {path} (length {len(path) - 1})")

In [ ]:
# BFS on a Grid: Number of Islands
print("\nBFS ON GRID: Number of Islands")
print("=" * 60)

def num_islands(grid):
    """
    Count islands (connected groups of '1's) in a grid.
    Time: O(rows * cols), Space: O(min(rows, cols))
    """
    if not grid:
        return 0

    rows, cols = len(grid), len(grid[0])
    islands = 0

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                islands += 1
                # BFS to mark entire island as visited
                queue = deque([(r, c)])
                grid[r][c] = "0"

                while queue:
                    row, col = queue.popleft()
                    for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                        nr, nc = row + dr, col + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                            queue.append((nr, nc))
                            grid[nr][nc] = "0"

    return islands

grid = [
    ["1", "1", "0", "0", "0"],
    ["1", "1", "0", "0", "0"],
    ["0", "0", "1", "0", "0"],
    ["0", "0", "0", "1", "1"],
]
print("  Grid:")
for row in grid:
    print(f"    {row}")

result = num_islands([row[:] for row in grid])
print(f"  Number of islands: {result}")
print("\n  Each '1' connected to adjacent '1's = one island.")
print("  BFS marks entire island as visited so we don't double-count.")

---
# Section 3: DFS -- Depth-First Search

## What is DFS?

Explore as **deep** as possible before backtracking.
Uses a **stack** (or recursion, which uses the call stack).

```
Starting from A (going deep first):
A -> B -> C (dead end, backtrack)
         -> E -> D (already visited, backtrack)
         -> D (from A, already visited)
```

## DFS Templates

```python
# Recursive DFS
def dfs(graph, node, visited):
    visited.add(node)
    process(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs(graph, neighbor, visited)

# Iterative DFS (using explicit stack)
def dfs_iterative(graph, start):
    visited = set()
    stack = [start]
    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            process(node)
            for neighbor in graph[node]:
                if neighbor not in visited:
                    stack.append(neighbor)
```

## When to Use DFS

- **Connected components** (find all reachable nodes)
- **Cycle detection**
- **Path existence** (does a path exist from A to B?)
- **Topological sort**
- **Backtracking** (find ALL paths, permutations)

In [ ]:
# DFS implementation (recursive and iterative)
print("DFS -- Depth-First Search")
print("=" * 60)

def dfs_recursive(graph, start, visited=None):
    """DFS using recursion. Time: O(V + E), Space: O(V)."""
    if visited is None:
        visited = set()
    visited.add(start)
    order = [start]
    for neighbor in graph[start]:
        if neighbor not in visited:
            order.extend(dfs_recursive(graph, neighbor, visited))
    return order

def dfs_iterative(graph, start):
    """DFS using explicit stack. Time: O(V + E), Space: O(V)."""
    visited = set()
    stack = [start]
    order = []
    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            order.append(node)
            # Add neighbors in reverse for consistent order
            for neighbor in reversed(graph[node]):
                if neighbor not in visited:
                    stack.append(neighbor)
    return order

graph = {
    "A": ["B", "D"],
    "B": ["A", "C", "E"],
    "C": ["B"],
    "D": ["A", "E"],
    "E": ["B", "D"],
}
print(f"  DFS recursive from A: {dfs_recursive(graph, 'A')}")
print(f"  DFS iterative from A: {dfs_iterative(graph, 'A')}")
print()
print("  DFS goes deep: A -> B -> C (dead end) -> E -> D")
print("  BFS goes wide: A -> B, D -> C, E")

In [ ]:
# DFS Application: Connected Components
print("\nDFS: CONNECTED COMPONENTS")
print("=" * 60)

def count_components(n, edges):
    """
    Count connected components in an undirected graph.
    Time: O(V + E), Space: O(V + E)
    """
    # Build adjacency list
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)

    visited = set()
    components = 0

    for node in range(n):
        if node not in visited:
            components += 1
            # DFS to mark all nodes in this component
            stack = [node]
            while stack:
                curr = stack.pop()
                if curr not in visited:
                    visited.add(curr)
                    for neighbor in graph[curr]:
                        if neighbor not in visited:
                            stack.append(neighbor)

    return components

n = 6
edges = [(0, 1), (1, 2), (3, 4)]
print(f"  Nodes: {n}, Edges: {edges}")
print(f"  Connected components: {count_components(n, edges)}")
print("  Component 1: {0, 1, 2}")
print("  Component 2: {3, 4}")
print("  Component 3: {5} (isolated)")

---
# Section 4: Topological Sort

## What is Topological Sort?

A linear ordering of nodes in a **DAG** (Directed Acyclic Graph) such that
for every directed edge u -> v, node u comes before v in the ordering.

```
Example DAG (course prerequisites):
  Math -> Physics -> Engineering
  Math -> CS -> AI
  English -> CS

Topological order: English, Math, Physics, CS, Engineering, AI
(Many valid orderings exist)
```

## Why It Matters for Data Engineers

- **Pipeline scheduling**: Tasks must run after their dependencies
- **Spark stage ordering**: Stages execute in topological order
- **Build systems**: Compile dependencies before dependents
- **Data lineage**: Process upstream tables before downstream

## Two Approaches

1. **Kahn's algorithm** (BFS-based): Start from nodes with no dependencies
2. **DFS-based**: Process node after all descendants are done

In [ ]:
# Topological Sort: Kahn's Algorithm (BFS)
print("TOPOLOGICAL SORT -- Kahn's Algorithm (BFS)")
print("=" * 60)

def topological_sort_kahn(graph, all_nodes):
    """
    Kahn's algorithm: BFS from nodes with 0 in-degree.
    Time: O(V + E), Space: O(V)
    Returns: sorted order, or empty if cycle exists.
    """
    # Calculate in-degree for each node
    in_degree = {node: 0 for node in all_nodes}
    for node in graph:
        for neighbor in graph[node]:
            in_degree[neighbor] = in_degree.get(neighbor, 0) + 1

    # Start with nodes that have no dependencies (in-degree = 0)
    queue = deque([node for node in all_nodes if in_degree[node] == 0])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in graph.get(node, []):
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    # If we processed all nodes, valid ordering exists
    if len(order) == len(all_nodes):
        return order
    return []  # Cycle detected!

# Pipeline dependency example
pipeline = {
    "ingest_raw": ["clean_data"],
    "clean_data": ["transform", "validate"],
    "transform": ["build_gold"],
    "validate": ["build_gold"],
    "build_gold": ["report"],
    "report": [],
}
all_tasks = ["ingest_raw", "clean_data", "transform", "validate", "build_gold", "report"]

order = topological_sort_kahn(pipeline, all_tasks)
print("  Pipeline dependencies:")
for task, deps in pipeline.items():
    if deps:
        print(f"    {task} -> {deps}")
print(f"\n  Execution order: {order}")
print("  This is exactly what Airflow/Lakeflow Jobs computes!")

In [ ]:
# Topological Sort: DFS-based
print("\nTOPOLOGICAL SORT -- DFS-based")
print("=" * 60)

def topological_sort_dfs(graph, all_nodes):
    """
    DFS-based: process node after all its descendants.
    Time: O(V + E), Space: O(V)
    """
    visited = set()
    order = []

    def dfs(node):
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor)
        order.append(node)  # Add AFTER processing all descendants

    for node in all_nodes:
        if node not in visited:
            dfs(node)

    return order[::-1]  # Reverse for correct order

# Course schedule example
courses = {
    "Algebra": ["Calculus", "Linear Algebra"],
    "Calculus": ["Physics", "ML"],
    "Linear Algebra": ["ML"],
    "Physics": [],
    "ML": ["AI"],
    "AI": [],
    "English": ["Technical Writing"],
    "Technical Writing": [],
}
all_courses = list(courses.keys())

order = topological_sort_dfs(courses, all_courses)
print("  Course prerequisites:")
for course, prereqs in courses.items():
    if prereqs:
        print(f"    {course} -> enables {prereqs}")
print(f"\n  Study order: {order}")

In [ ]:
# Cycle Detection (Course Schedule problem)
print("\nCYCLE DETECTION: Can you finish all courses?")
print("=" * 60)

def can_finish_courses(num_courses, prerequisites):
    """
    Detect if cycle exists in course prerequisites.
    Uses DFS with 3 states: unvisited, in-progress, done.
    Time: O(V + E), Space: O(V)
    """
    graph = defaultdict(list)
    for course, prereq in prerequisites:
        graph[prereq].append(course)

    # 0 = unvisited, 1 = in-progress (current DFS path), 2 = done
    state = [0] * num_courses

    def has_cycle(node):
        if state[node] == 1:
            return True  # Back edge = cycle!
        if state[node] == 2:
            return False  # Already processed

        state[node] = 1  # Mark in-progress
        for neighbor in graph[node]:
            if has_cycle(neighbor):
                return True
        state[node] = 2  # Mark done
        return False

    for i in range(num_courses):
        if has_cycle(i):
            return False
    return True

# No cycle
prereqs = [(1, 0), (2, 1), (3, 2)]
print(f"  Courses: 4, Prerequisites: {prereqs}")
print(f"  Can finish: {can_finish_courses(4, prereqs)}")

# Has cycle: 0->1->2->0
prereqs_cycle = [(1, 0), (2, 1), (0, 2)]
print(f"  Courses: 3, Prerequisites: {prereqs_cycle}")
print(f"  Can finish: {can_finish_courses(3, prereqs_cycle)} (cycle detected!)")

---
# Section 5: Union-Find (Disjoint Set Union)

## What is Union-Find?

A data structure to efficiently:
- **Find**: Which group does element x belong to?
- **Union**: Merge the groups containing x and y

## Operations

| Operation | Time (with optimizations) |
|-----------|--------------------------|
| Find | O(alpha(n)) ~ O(1) amortized |
| Union | O(alpha(n)) ~ O(1) amortized |
| Connected? | O(alpha(n)) ~ O(1) amortized |

## When to Use Union-Find

- **Connected components** (alternative to DFS)
- **Detect cycles** in undirected graph
- **Kruskal's minimum spanning tree**
- **Network connectivity** questions
- Whenever you need to group elements and query "are these two in the same group?"

## Optimizations

1. **Path compression**: Point each node directly to root during Find
2. **Union by rank**: Attach smaller tree under root of larger tree

In [ ]:
# Union-Find implementation
print("UNION-FIND (Disjoint Set Union)")
print("=" * 60)

class UnionFind:
    """
    Union-Find with path compression and union by rank.
    Nearly O(1) per operation (amortized).
    """
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.components = n

    def find(self, x):
        """Find root of x with path compression."""
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]

    def union(self, x, y):
        """Union the sets containing x and y. Returns True if merged."""
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False  # Already in same set

        # Union by rank: attach smaller tree under larger
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1

        self.components -= 1
        return True

    def connected(self, x, y):
        """Check if x and y are in the same set."""
        return self.find(x) == self.find(y)

# Demo
uf = UnionFind(7)
print("  7 nodes (0-6), initially each in own group:")
print(f"  Components: {uf.components}")

edges = [(0, 1), (1, 2), (3, 4), (5, 6), (4, 5)]
for u, v in edges:
    uf.union(u, v)
    print(f"  Union({u}, {v}) -> components: {uf.components}")

print(f"\n  Connected(0, 2): {uf.connected(0, 2)}")
print(f"  Connected(0, 3): {uf.connected(0, 3)}")
print(f"  Connected(3, 6): {uf.connected(3, 6)}")

In [ ]:
# Union-Find Application: Number of Islands (alternative to BFS)
print("\nUNION-FIND: Number of Connected Components from Edges")
print("=" * 60)

def count_components_uf(n, edges):
    """Count connected components using Union-Find."""
    uf = UnionFind(n)
    for u, v in edges:
        uf.union(u, v)
    return uf.components

# Same problem from earlier, now with Union-Find
n = 6
edges = [(0, 1), (1, 2), (3, 4)]
print(f"  Nodes: {n}, Edges: {edges}")
print(f"  Components (Union-Find): {count_components_uf(n, edges)}")
print("  Same answer as DFS, but Union-Find is better for streaming edges!")
print()
print("  DE Connection: As new data relationships arrive (streaming),")
print("  Union-Find efficiently tracks connected components without")
print("  re-running DFS from scratch.")

---
# Section 6: Dijkstra's Algorithm -- Shortest Path with Weights

## When BFS Isn't Enough

BFS finds shortest path in **unweighted** graphs (all edges cost 1).
For **weighted** graphs (edges have different costs), use Dijkstra's.

## How Dijkstra Works

1. Start at source with distance 0, all others = infinity
2. Use a **min-heap** to always process the closest unvisited node
3. For each neighbor, check if going through current node is shorter
4. Repeat until all nodes are processed

## Time Complexity: O((V + E) log V) with a heap

## DE Connection

- **Network latency**: Find fastest route between data centers
- **Cost optimization**: Cheapest way to move data between regions
- **ETL optimization**: Minimize total processing time across pipeline stages

In [ ]:
# Dijkstra's Algorithm
import heapq

print("DIJKSTRA'S ALGORITHM -- Shortest Path (Weighted)")
print("=" * 60)

def dijkstra(graph, start):
    """
    Find shortest distances from start to all other nodes.
    graph: {node: [(neighbor, weight), ...]}
    Time: O((V + E) log V), Space: O(V)
    """
    distances = {node: float("inf") for node in graph}
    distances[start] = 0
    heap = [(0, start)]  # (distance, node)
    visited = set()

    while heap:
        dist, node = heapq.heappop(heap)

        if node in visited:
            continue
        visited.add(node)

        for neighbor, weight in graph[node]:
            new_dist = dist + weight
            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                heapq.heappush(heap, (new_dist, neighbor))

    return distances

# Weighted graph example
graph = {
    "A": [("B", 4), ("C", 2)],
    "B": [("A", 4), ("C", 1), ("D", 5)],
    "C": [("A", 2), ("B", 1), ("D", 8), ("E", 10)],
    "D": [("B", 5), ("C", 8), ("E", 2)],
    "E": [("C", 10), ("D", 2)],
}

distances = dijkstra(graph, "A")
print("  Weighted graph:")
print("    A -4- B -5- D")
print("    |  /  |   / |")
print("    2 1   8  2  |")
print("    |/    | /   |")
print("    C ----+- E")
print("      10")
print()
print(f"  Shortest distances from A:")
for node, dist in sorted(distances.items()):
    print(f"    A -> {node}: {dist}")

---
# Section 7: Your Turn -- Practice Problems

In [ ]:
# ============================================
# YOUR TURN: Problem 1 -- Clone Graph
# ============================================
# Given a reference to a node in a connected undirected graph,
# return a deep copy (clone) of the graph.
# Each node has val (int) and neighbors (list of nodes).
#
# Hint: Use BFS or DFS with a hash map {original_node: cloned_node}

# (This problem uses a Node class -- implement the logic conceptually)
# def clone_graph(node):
#     pass
print("Problem 1: Clone Graph")
print("  Approach: BFS/DFS + hash map to track old->new node mapping")
print("  Time: O(V + E), Space: O(V)")


In [ ]:
# ============================================
# YOUR TURN: Problem 2 -- Course Schedule Order
# ============================================
# Given numCourses and prerequisites, return an ordering
# to take all courses (topological sort), or [] if impossible.
#
# Example: numCourses=4, prereqs=[[1,0],[2,0],[3,1],[3,2]]
#          Output: [0, 1, 2, 3] or [0, 2, 1, 3]
#
# Hint: Topological sort with cycle detection (Kahn's algorithm)

def find_order(num_courses, prerequisites):
    pass  # Your solution here


In [ ]:
# ============================================
# YOUR TURN: Problem 3 -- Word Ladder
# ============================================
# Given begin_word, end_word, and a word_list, find the shortest
# transformation sequence length from begin to end.
# Only one letter can change at a time, and each intermediate word
# must be in word_list.
#
# Example: "hit" -> "cog", word_list=["hot","dot","dog","lot","log","cog"]
#          Output: 5 (hit -> hot -> dot -> dog -> cog)
#
# Hint: This is BFS! Each word is a node, edges connect words
#       that differ by one letter.

def ladder_length(begin_word, end_word, word_list):
    pass  # Your solution here


---
# Solutions

In [ ]:
# SOLUTION 2: Course Schedule Order (Topological Sort)
print("SOLUTION 2: Course Schedule Order")
print("=" * 60)

def find_order(num_courses, prerequisites):
    """
    Topological sort using Kahn's algorithm.
    Time: O(V + E), Space: O(V + E)
    """
    graph = defaultdict(list)
    in_degree = [0] * num_courses

    for course, prereq in prerequisites:
        graph[prereq].append(course)
        in_degree[course] += 1

    # Start with courses that have no prerequisites
    queue = deque([i for i in range(num_courses) if in_degree[i] == 0])
    order = []

    while queue:
        course = queue.popleft()
        order.append(course)
        for next_course in graph[course]:
            in_degree[next_course] -= 1
            if in_degree[next_course] == 0:
                queue.append(next_course)

    return order if len(order) == num_courses else []

tests = [
    (4, [[1,0],[2,0],[3,1],[3,2]]),
    (2, [[1,0]]),
    (2, [[1,0],[0,1]]),  # Cycle!
]
for n, prereqs in tests:
    result = find_order(n, prereqs)
    print(f"  courses={n}, prereqs={prereqs}")
    print(f"  Order: {result}")
    print()

In [ ]:
# SOLUTION 3: Word Ladder (BFS)
print("SOLUTION 3: Word Ladder (BFS)")
print("=" * 60)

def ladder_length(begin_word, end_word, word_list):
    """
    BFS where each word is a node, edges connect words differing by 1 letter.
    Time: O(n * m^2) where n = words, m = word length
    Space: O(n * m)
    """
    word_set = set(word_list)
    if end_word not in word_set:
        return 0

    queue = deque([(begin_word, 1)])
    visited = set([begin_word])

    while queue:
        word, steps = queue.popleft()

        for i in range(len(word)):
            for c in "abcdefghijklmnopqrstuvwxyz":
                next_word = word[:i] + c + word[i+1:]
                if next_word == end_word:
                    return steps + 1
                if next_word in word_set and next_word not in visited:
                    visited.add(next_word)
                    queue.append((next_word, steps + 1))

    return 0  # No path

result = ladder_length("hit", "cog", ["hot","dot","dog","lot","log","cog"])
print(f"  'hit' -> 'cog': {result} steps")
print("  Path: hit -> hot -> dot -> dog -> cog")
print()
print("  Key insight: Transform the problem into a GRAPH problem!")
print("  Nodes = words, Edges = words differing by 1 letter")
print("  Shortest path = BFS")

---
# Section 8: Summary

## Algorithm Choice Guide

| Problem Type | Algorithm | Time |
|-------------|-----------|------|
| Shortest path (unweighted) | BFS | O(V + E) |
| Shortest path (weighted) | Dijkstra | O((V+E) log V) |
| Explore all nodes/paths | DFS | O(V + E) |
| Connected components | DFS or Union-Find | O(V + E) |
| Dependency ordering | Topological Sort | O(V + E) |
| Cycle detection (directed) | DFS with states | O(V + E) |
| Cycle detection (undirected) | Union-Find | O(V + E) |
| Dynamic connectivity | Union-Find | ~O(1) per query |

## BFS vs DFS

| | BFS | DFS |
|---|---|---|
| Data structure | Queue | Stack/Recursion |
| Explores | Level by level | As deep as possible |
| Finds | Shortest path (unweighted) | All paths, cycles |
| Memory | O(width of level) | O(depth) |
| Use when | Need MINIMUM steps/distance | Need to explore ALL possibilities |

## LeetCode Problems

**Medium:** Number of Islands (#200), Course Schedule (#207, #210),
           Clone Graph (#133), Word Ladder (#127), Pacific Atlantic Water Flow (#417)
**Hard:** Word Ladder II (#126), Alien Dictionary (#269), Network Delay (#743)

---
*Next: DSA_06 -- Recursion, Backtracking & Dynamic Programming*

In [ ]:
print("KEY TAKEAWAYS FROM DSA_05")
print("=" * 50)
print()
print("1. Graphs = nodes + edges. Adjacency list is most common.")
print("2. BFS (queue): shortest path, level-by-level, minimum steps")
print("3. DFS (stack/recursion): explore all paths, connected components")
print("4. Topological sort: dependency ordering (Kahn's or DFS)")
print("5. Union-Find: dynamic connectivity, ~O(1) per operation")
print("6. Dijkstra: shortest path with weights (heap-based)")
print()
print("DE tip: Pipeline DAGs, Spark stages, data lineage -- all graphs!")
print("If you can model it as a graph, you can solve it with BFS/DFS.")
print()
print("Next: DSA_06 -- Recursion, Backtracking & Dynamic Programming")